# `parse_question` — Brique 2 : parser une question utilisateur

Référence : [`docs/06_question_parsing.md`](../docs/06_question_parsing.md).

Module testé : [`src/docpipeline/question_parsing/question_parsing.py`](../src/docpipeline/question_parsing/question_parsing.py).

## Ce que fait `parse_question(question, document_type)`

Transforme une question utilisateur en un JSON structuré (`list[dict]`, drop-in compat avec `src/question/understand_question`). Chaque entrée contient :

| Section | Champs |
|---|---|
| `retrieval`  | `main_query`, `page_hint`, `section_hint`, `layout_hint`, `document_hint`, `anchor_keywords` |
| `generation` | `original_question`, `format_constraint`, `disambiguation`, `must_distinguish` |
| `_meta`      | `intent`, `document_type`, `bricks_active` (traçabilité) |

Le JSON ne contient **que des champs populés** (pas de `null`). Les briques actives dépendent du `document_type` (preset : pas de `page_hint` en Word, etc.). Approche : regex / heuristique uniquement (pas de LLM, conforme à `CLAUDE.md`).

Question de démonstration ci-dessous : un cas riche qui couvre 4 briques (`anchor_keywords` `L131-1`, `page_hint` 5, `section_hint` `limits`, `disambiguation` `deductible`).

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import json
from docpipeline.question_parsing import parse_question

QUESTION = "What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies."

plan = parse_question(QUESTION, document_type='pdf')
print(json.dumps(plan, indent=2, ensure_ascii=False))

[
  {
    "retrieval": {
      "main_query": "What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies.",
      "anchor_keywords": [
        "L131-1",
        "Article L131-1"
      ],
      "page_hint": 5,
      "section_hint": "limits"
    },
    "generation": {
      "original_question": "What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies.",
      "disambiguation": "L'utilisateur demande la notion principale, PAS : deductible. Ces concepts apparaissent souvent ensemble dans les passages remontés — n'extraire que la notion principale.",
      "must_distinguish": [
        "deductible"
      ]
    },
    "_meta": {
      "intent": "extract",
      "document_type": "pdf",
      "bricks_active": [
        "anchor_keywords",
        "page_hint",
        "section_hint",
        "disambiguation"
      ]
    }
  }
]
